# GraphKV 第四阶段：对应关系压力测试与可选 FP32 对照

前三阶段支持：图边确实改变对应 target 的缓存；普通顺序输入较稳定；图计算下 LYRA 单问反复被回答成 copper；BF16 分块与完整图计算有数值差异。

本阶段用 **8 个受控变体** 检查错误是否跟项目名、供应商名、材料名、source/target 顺序有关。保留原五节点图与原 legacy 提示格式；主实验全部在物理 GPU 3 上运行 BF16/eager。

真实 8B FP32 权重约需 30 GiB，单张 24GB 卡装不下。可选 FP32 对照放在 CPU，默认关闭，不启用多卡、量化或 offload。若开启，要求 /proc/meminfo 显示至少 48 GiB 可用 RAM，并且先释放 GPU 模型。CPU 计算可能较慢，未执行此项时不能作出“高精度已验证”的结论。

交付时只完成静态与无模型逻辑检查，实际 GPU/CPU 推理需在你的 Linux 环境执行。建议重启内核，释放旧 Notebook 模型。


## 1. 沿用路径、图与单卡环境


In [1]:
import os, sys
from pathlib import Path
GPU_ID = "3"
REPO_DIR = Path("/home/lyh/hcy_work/GraphKV_refactor_v5")  # 改为含两个 graph_kv 模块的目录
MODEL_PATH = Path("/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT")
BACKEND = "eager"  # 默认不依赖 flash-attn；也可用 eager / flash_attention_2
MAX_NEW_TOKENS = 80
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化，请重启内核，再从本格运行。")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# GitHub 仓库外层可能还有一层同名目录。
if not (REPO_DIR / "graph_kv_cache.py").is_file():
    nested = REPO_DIR / "GraphKV_refactor_v5"
    if (nested / "graph_kv_cache.py").is_file():
        REPO_DIR = nested
assert (REPO_DIR / "graph_kv_cache.py").is_file(), f"请修改 REPO_DIR: {REPO_DIR}"
assert (MODEL_PATH / "config.json").is_file(), f"找不到本地模型: {MODEL_PATH}"
sys.path.insert(0, str(REPO_DIR.resolve()))
print("模块目录:", REPO_DIR, "模型目录:", MODEL_PATH, "物理 GPU:", GPU_ID, sep="\n")


模块目录:
/home/lyh/hcy_work/GraphKV_refactor_v5
模型目录:
/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT
物理 GPU:
3


In [2]:
import json, inspect, gc, time
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import greedy_generate_graphkv, graphkv_next_logits, build_graphkv_from_ids
from IPython.display import display, HTML
assert transformers.__version__ == "4.50.0", "v5 adapter 要求 transformers==4.50.0，请选择原有兼容内核。"
assert torch.cuda.is_available(), "当前内核没有可用 CUDA"
assert torch.cuda.device_count() == 1, "单卡屏蔽未生效，请重启内核"
assert torch.cuda.is_bf16_supported(), "此测试默认使用 BF16"
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("实际导入:", inspect.getfile(GraphKVCache))
print("GPU:", torch.cuda.get_device_name(0))
def memory(label):
    free, total = torch.cuda.mem_get_info(0)
    print(f"{label}: free={free/2**30:.2f}/{total/2**30:.2f} GiB; "
          f"allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB; "
          f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
memory("加载前")


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 transformers: 4.50.0
实际导入: /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py
GPU: NVIDIA GeForce RTX 4090
加载前: free=23.26/23.64 GiB; allocated=0.00 GiB; peak=0.00 GiB


In [3]:
SOURCE_IDS = ["4", "5"]
TARGET_IDS = ["1", "2", "3"]
EDGES = [("4", "1"), ("4", "3"), ("5", "2"), ("5", "3")]
source_texts = {
    "4": "[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n",
    "5": "[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n",
}
target_texts = {
    "1": "[Document 1] Project ORION uses the material supplied by ALPHA.\n",
    "2": "[Document 2] Project LYRA uses the material supplied by BETA.\n",
    "3": "[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n",
}
PREFIX = "<|user|>\nAnswer using the following documents. Give a short factual answer.\n"
QUERY = "\nQuestion: What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?\n<|assistant|>\n"
assert set(source_texts).isdisjoint(target_texts)
assert all(s in source_texts and t in target_texts for s,t in EDGES)
print("边:", EDGES)
for n, text in {**source_texts, **target_texts}.items():
    print(n, "source" if n in SOURCE_IDS else "target", text)
print("target <- sources:", {t:[s for s,d in EDGES if d==t] for t in TARGET_IDS})
display(HTML("<pre>4 ─────► 1\n│\n└──────► 3 ◄──────┐\n                   │\n5 ─────► 2         │\n└──────────────────┘</pre>"))


边: [('4', '1'), ('4', '3'), ('5', '2'), ('5', '3')]
4 source [Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.

5 source [Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.

1 target [Document 1] Project ORION uses the material supplied by ALPHA.

2 target [Document 2] Project LYRA uses the material supplied by BETA.

3 target [Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.

target <- sources: {'1': ['4'], '2': ['5'], '3': ['4', '5']}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
config = AutoConfig.from_pretrained(str(MODEL_PATH), local_files_only=True)
assert config.model_type == "llama"
rope = config.rope_scaling or {}
assert rope.get("rope_type", rope.get("type", "default")) in ("default", "llama3", "linear")
def tokenize(text):
    return tokenizer(text, return_tensors="pt", add_special_tokens=False,
                     truncation=False).input_ids
node_inputs = {n:tokenize(t) for n,t in {**source_texts, **target_texts}.items()}
prefix_ids, query_ids = tokenize(PREFIX), tokenize(QUERY)
lengths = {n:int(x.shape[1]) for n,x in node_inputs.items()}
P = int(prefix_ids.shape[1])
L = max(lengths.values())
Q = int(query_ids.shape[1])
assert P + 2*L + Q + MAX_NEW_TOKENS <= config.max_position_embeddings
print("P, L, Q =", P, L, Q)
print("node | kind | tokens | logical range [start,end)")
for n in SOURCE_IDS + TARGET_IDS:
    start = P if n in SOURCE_IDS else P+L
    print(n, "source" if n in SOURCE_IDS else "target", lengths[n], (start,start+lengths[n]))
print("query logical:", (P+2*L, P+2*L+Q))
print("query physical start:", P+sum(lengths.values()))
print("Tulu markers:", {x:tokenizer.encode(x, add_special_tokens=False) for x in ["<|user|>","<|assistant|>"]})


P, L, Q = 17 21 31
node | kind | tokens | logical range [start,end)
4 source 20 (17, 37)
5 source 21 (17, 38)
1 target 15 (38, 53)
2 target 16 (38, 54)
3 target 18 (38, 56)
query logical: (59, 90)
query physical start: 107
Tulu markers: {'<|user|>': [27, 91, 882, 91, 29], '<|assistant|>': [27, 91, 78191, 91, 29]}


In [5]:
from graph_kv_adapter import build_graphkv_from_ids
import re, copy, hashlib
BACKEND="eager"
MAX_NEW_TOKENS=64
RUN_CPU_FP32=True  # 有充足 CPU RAM 且愿意等待时改 True
CPU_THREADS=8
OUTPUT_DIR=Path.cwd()/"graphkv_stage4_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
ORIGINAL={"source_texts":copy.deepcopy(source_texts),"target_texts":copy.deepcopy(target_texts),
 "source_order":SOURCE_IDS.copy(),"target_order":TARGET_IDS.copy(),
 "projects":["ORION","LYRA","NOVA"],"materials":["copper","silver"]}
CONDITIONS={"graph_shared":("graph","shared"),
 "graph_continuous":("graph","continuous"),
 "ordinary_sequence":("causal","continuous")}
def query_text(question):
    return "\nQuestion: "+question+"\n<|assistant|>\n"
QUESTIONS={"ORION":"What material does ORION use? Answer with the material name only."}
results={"settings":{"model_path":str(MODEL_PATH),"repo_dir":str(REPO_DIR),
 "physical_gpu":GPU_ID,"backend":BACKEND,"dtype":"bfloat16","max_new_tokens":MAX_NEW_TOKENS,
 "torch":torch.__version__,"transformers":transformers.__version__,"prefix":PREFIX},
 "cases":[],"answers":[],"precision":[],"precision_status":"not_requested"}
def save_results():
    path=OUTPUT_DIR/"graphkv_stage4_results.json";tmp=path.with_suffix(".tmp")
    tmp.write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding="utf-8")
    tmp.replace(path)


## 2. 构造八个变体
每个变体只改一类因素：原例、交换铜/银、替换项目名、替换供应商名、替换材料名、反转 source、反转 target、同时反转两组节点。

材料交换后正确答案也交换。问题依旧分别读取 target 1、2、3 对应的项目。Document 编号保持不变，这轮尚不能排除编号影响。这里只有少量受控样本，结果用于定位，不能当作论文基准准确率。

评分既检查材料，也检查回答中是否出现不应该属于该项目的供应商。只要解释涉及错误供应商，即使材料正确也不会算关系检查通过；复杂否定表述仍需人工复核。


In [6]:
def replace_words(case,mapping):
    pat=re.compile(r"\b("+"|".join(re.escape(k) for k in mapping)+r")\b")
    for group in ["source_texts","target_texts"]:
        case[group]={n:pat.sub(lambda m:mapping[m.group(0)],text) for n,text in case[group].items()}
    return case
def make_cases():
    specs=[]
    for name in ["original","swap_materials","rename_projects","rename_suppliers",
                 "rename_materials","reverse_sources","reverse_targets","reverse_both"]:
        c=copy.deepcopy(ORIGINAL);c["name"]=name;c["suppliers"]=["ALPHA","BETA"]
        if name=="swap_materials":
            replace_words(c,{"copper":"silver","silver":"copper"});c["materials"]=["silver","copper"]
        elif name=="rename_projects":
            replace_words(c,{"ORION":"CEDAR","LYRA":"MAPLE","NOVA":"BIRCH"});c["projects"]=["CEDAR","MAPLE","BIRCH"]
        elif name=="rename_suppliers":
            replace_words(c,{"ALPHA":"DELTA","BETA":"GAMMA"});c["suppliers"]=["DELTA","GAMMA"]
        elif name=="rename_materials":
            replace_words(c,{"copper":"tin","silver":"zinc"});c["materials"]=["tin","zinc"]
        elif name=="reverse_sources":c["source_order"].reverse()
        elif name=="reverse_targets":c["target_order"].reverse()
        elif name=="reverse_both":c["source_order"].reverse();c["target_order"].reverse()
        a,b=c["materials"];s,t=c["suppliers"];p,q,r=c["projects"]
        c["questions"]={"1":f"What material does {p} use? Answer with the material name only.",
            "2":f"What material does {q} use? Answer with the material name only.",
            "3":f"Which two materials does {r} combine? Answer with the material names only."}
        c["expected"]={"1":[a],"2":[b],"3":[a,b]}
        c["allowed_suppliers"]={"1":[s],"2":[t],"3":[s,t]}
        specs.append(c)
    return specs
CASES=make_cases()
ALL_MATERIALS=["copper","silver","tin","zinc"]
ALL_SUPPLIERS=["ALPHA","BETA","DELTA","GAMMA"]
def evaluate(case,target,text):
    lower=text.lower()
    found={w for w in ALL_MATERIALS if re.search(r"\b"+w+r"\b",lower)}
    named={w for w in ALL_SUPPLIERS if re.search(r"\b"+w.lower()+r"\b",lower)}
    expected=set(case["expected"][target]);allowed=set(case["allowed_suppliers"][target])
    return {"found_materials":sorted(found),"material_pass":found==expected,
        "named_suppliers":sorted(named),"wrong_suppliers":sorted(named-allowed),
        "relation_check_pass":found==expected and not (named-allowed)}
def activate(case):
    global SOURCE_IDS,TARGET_IDS,node_inputs,P,L
    SOURCE_IDS=case["source_order"].copy();TARGET_IDS=case["target_order"].copy()
    texts={**case["source_texts"],**case["target_texts"]}
    node_inputs={n:tokenize(texts[n]) for n in SOURCE_IDS+TARGET_IDS}
    P=prefix_ids.shape[1];L=max(x.shape[1] for x in node_inputs.values())
for c in CASES:
    assert set(c["source_order"])=={"4","5"} and set(c["target_order"])=={"1","2","3"}
    print(c["name"],c["projects"],c["expected"],c["source_order"],c["target_order"])
assert evaluate(CASES[0],"1","BETA supplies copper")["relation_check_pass"]==False
assert evaluate(CASES[1],"1","Silver.")["material_pass"]
results["cases"]=CASES
print("生成次数:",len(CASES)*len(CONDITIONS)*3)


original ['ORION', 'LYRA', 'NOVA'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver']} ['4', '5'] ['1', '2', '3']
swap_materials ['ORION', 'LYRA', 'NOVA'] {'1': ['silver'], '2': ['copper'], '3': ['silver', 'copper']} ['4', '5'] ['1', '2', '3']
rename_projects ['CEDAR', 'MAPLE', 'BIRCH'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver']} ['4', '5'] ['1', '2', '3']
rename_suppliers ['ORION', 'LYRA', 'NOVA'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver']} ['4', '5'] ['1', '2', '3']
rename_materials ['ORION', 'LYRA', 'NOVA'] {'1': ['tin'], '2': ['zinc'], '3': ['tin', 'zinc']} ['4', '5'] ['1', '2', '3']
reverse_sources ['ORION', 'LYRA', 'NOVA'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver']} ['5', '4'] ['1', '2', '3']
reverse_targets ['ORION', 'LYRA', 'NOVA'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver']} ['4', '5'] ['3', '2', '1']
reverse_both ['ORION', 'LYRA', 'NOVA'] {'1': ['copper'], '2': ['silver'], '3': ['copper', 'silver

## 3. 计算与比较工具
沿用第三阶段已使用的显式 mask 完整图计算。生成逐步重新计算短上下文，不调用 v5 缓存拼接。原例另外运行 v5/完整图的数值对照；CPU FP32 与 GPU BF16 的对照会同时改变设备和精度，不能视为只改变精度的严格实验。


In [7]:
def make_layout(query_ids, attention="graph", positions="shared"):
    chunks=[("prefix",prefix_ids)] + [(n,node_inputs[n]) for n in SOURCE_IDS+TARGET_IDS] + [("query",query_ids)]
    ranges={}; offsets=0; pos=[]; tensors=[]
    for name,ids in chunks:
        length=int(ids.shape[1]); ranges[name]=(offsets,offsets+length)
        if positions=="continuous":start=offsets
        elif name=="prefix":start=0
        elif name in SOURCE_IDS:start=P
        elif name in TARGET_IDS:start=P+L
        else:start=P+2*L
        pos.extend(range(start,start+length)); offsets+=length; tensors.append(ids.cpu())
    allow=torch.zeros((offsets,offsets),dtype=torch.bool)
    if attention=="causal":
        allow=torch.ones_like(allow).tril()
    else:
        for name,ids in chunks:
            a,b=ranges[name]; allow[a:b,a:b]=torch.ones((b-a,b-a),dtype=torch.bool).tril()
            if name=="query":allow[a:b,:a]=True
            elif attention=="graph" and name in TARGET_IDS:
                for s,t in EDGES:
                    if t==name:
                        c,d=ranges[s];allow[a:b,c:d]=True
    return torch.cat(tensors,-1),torch.tensor([pos]),allow,ranges
@torch.inference_mode()
def dense_forward(model,query_ids,attention="graph",positions="shared",keep_cache=False):
    ids,pos,allow,ranges=make_layout(query_ids,attention,positions)
    device=model.get_input_embeddings().weight.device
    mask=torch.zeros((1,1,ids.shape[1],ids.shape[1]),dtype=next(model.parameters()).dtype,device=device)
    mask.masked_fill_(~allow.to(device)[None,None],torch.finfo(mask.dtype).min)
    out=model(ids.to(device),position_ids=pos.to(device),
        cache_position=torch.arange(ids.shape[1],device=device),attention_mask=mask,
        use_cache=keep_cache,return_dict=True)
    return out,ranges
def check_mask_contract():
    q=tokenize(query_text(QUESTIONS["ORION"]))
    _,pos,a,r=make_layout(q)
    for target in TARGET_IDS:
        x,y=r[target]
        for source in SOURCE_IDS:
            u,v=r[source]
            assert bool(a[x:y,u:v].all())==((source,target) in EDGES)
        u,v=r["prefix"];assert not bool(a[x:y,u:v].any())
    x,y=r["query"];assert bool(a[x:y,:x].all())
    assert pos[0,x].item()==P+2*L
    _,pc,ac,rc=make_layout(q,"graph","continuous")
    assert torch.equal(a,ac) and r==rc
    assert torch.equal(pc,torch.arange(pc.shape[1]).unsqueeze(0))
    print("PASS: 4→1、5→2、4/5→3、prefix 隔离、query 可见与位置控制")
check_mask_contract()


PASS: 4→1、5→2、4/5→3、prefix 隔离、query 可见与位置控制


In [8]:
def difference(a,b):
    d=(a.detach().float()-b.detach().float()).abs()
    return {"max_abs":d.max().item(),"mean_abs":d.mean().item()}
def compare_logits(a,b):
    a=a.detach().float();b=b.detach().float()
    pa=a.softmax(-1);pb=b.softmax(-1)
    la=a.log_softmax(-1);lb=b.log_softmax(-1)
    row=difference(a,b)
    row.update({"KL_a_b":(pa*(la-lb)).sum(-1).mean().item(),
        "prob_L1":(pa-pb).abs().sum(-1).mean().item(),
        "argmax_a":a.argmax(-1).item(),"argmax_b":b.argmax(-1).item(),
        "top5_a":a[0].topk(5).indices.tolist(),"top5_b":b[0].topk(5).indices.tolist()})
    return row
def cache_hash(g):
    h=hashlib.sha256()
    for n in SOURCE_IDS+TARGET_IDS:
        for attr in ["key_cache","value_cache"]:
            for x in getattr(g.nodes[n],attr):
                h.update(x.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
def eos_ids(model):
    ids=model.generation_config.eos_token_id
    if ids is None:ids=tokenizer.eos_token_id
    return set([] if ids is None else ([ids] if isinstance(ids,int) else ids))
@torch.inference_mode()
def dense_generate(model,question,attention,positions):
    ids=tokenize(query_text(question)); tokens=[]; stopped=False
    eos=eos_ids(model)
    for _ in range(MAX_NEW_TOKENS):
        out,_=dense_forward(model,ids,attention,positions)
        token=out.logits[0,-1].argmax().item();del out
        tokens.append(token)
        if token in eos:stopped=True;break
        ids=torch.cat([ids,torch.tensor([[token]],dtype=torch.long)],-1)
    return {"answer":tokenizer.decode(tokens,skip_special_tokens=True),"token_ids":tokens,
        "eos":stopped,"hit_limit":len(tokens)==MAX_NEW_TOKENS and not stopped}
def rough_check(project,text):
    if project=="COMBINED":return None
    found={w for w in ["copper","silver"] if re.search(r"\b"+w+r"\b",text.lower())}
    expected={"ORION":{"copper"},"LYRA":{"silver"},"NOVA":{"copper","silver"}}
    return found==expected[project]
@torch.inference_mode()
def candidate_score(model,question,candidate,attention,positions):
    # 使用与自由生成相同的问题；包括大小写和空格变体，不将它们的概率直接相加。
    q=tokenize(query_text(question));a=tokenize(candidate)
    ids=torch.cat([q,a],-1)
    out,ranges=dense_forward(model,ids,attention,positions)
    start=ranges["query"][0]+q.shape[1]-1
    lp=out.logits[:,start:start+a.shape[1]].float().log_softmax(-1)
    vals=lp.gather(-1,a.to(lp.device).unsqueeze(-1)).squeeze(-1)[0]
    return {"candidate":candidate,"token_ids":a[0].tolist(),
        "sum_logp":vals.sum().item(),"per_token_logp":vals.tolist()}


In [9]:
@torch.inference_mode()
def oracle_comparison(model,device_label):
    case=CASES[0];activate(case)
    g=build_graphkv_from_ids(model=model,
        source_inputs={n:node_inputs[n] for n in SOURCE_IDS},
        target_inputs={n:node_inputs[n] for n in TARGET_IDS},edges=EDGES,
        prefix_input_ids=prefix_ids,chunk_span=L)
    before=cache_hash(g);measurements=[]
    for target in ["1","2","3"]:
        q=tokenize(query_text(case["questions"][target]))
        out,ranges=dense_forward(model,q,keep_cache=True)
        ref=out.logits[:,-1].detach().float()
        layer_diffs=[]
        for n in SOURCE_IDS+TARGET_IDS:
            a,b=ranges[n]
            for attr in ["key_cache","value_cache"]:
                for layer,(x,y) in enumerate(zip(getattr(g.nodes[n],attr),getattr(out.past_key_values,attr))):
                    row=difference(x,y[:,:,a:b,:]);row.update(node=n,kind=attr,layer=layer);layer_diffs.append(row)
        del out
        chunk=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q)
        repeated=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q)
        reverse=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q,
            source_ids=SOURCE_IDS[::-1],target_ids=TARGET_IDS[::-1])
        measurements.append({"mode":device_label,"target":target,
            "v5_vs_dense":compare_logits(chunk,ref),
            "v5_repeat":compare_logits(chunk,repeated),
            "v5_reverse":compare_logits(chunk,reverse),"per_layer_KV":layer_diffs})
        print("对照",device_label,target,measurements[-1]["v5_vs_dense"],flush=True)
        del ref,chunk,repeated,reverse
    assert cache_hash(g)==before
    del g
    return measurements


## 4. 单卡 BF16 主实验
每条回答保存实际文本、预期材料、关系检查、EOS 和是否被长度截断。每条结果立即写 JSON。source 顺序改变也会影响物理排列和连续位置；因此不能把顺序效应全部归因于共享位置。


In [10]:
torch.cuda.reset_peak_memory_stats()
model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
    torch_dtype=torch.bfloat16,device_map={"":0},low_cpu_mem_usage=True,
    attn_implementation=BACKEND).eval()
assert {str(p.device) for p in model.parameters()}=={"cuda:0"}
results["precision"].extend(oracle_comparison(model,"gpu_bf16"));save_results()
for case in CASES:
    activate(case)
    for condition,(attention,positions) in CONDITIONS.items():
        for target,question in case["questions"].items():
            torch.cuda.synchronize();start=time.perf_counter()
            row=dense_generate(model,question,attention,positions)
            torch.cuda.synchronize()
            row.update({"case":case["name"],"condition":condition,"target":target,
                "project":case["projects"][int(target)-1],"question":question,
                "expected_materials":case["expected"][target],"seconds":time.perf_counter()-start})
            row.update(evaluate(case,target,row["answer"]))
            results["answers"].append(row);save_results()
            print(case["name"],condition,target,repr(row["answer"]),
                  "material=",row["material_pass"],"relation=",row["relation_check_pass"],flush=True)
results["peak_gpu_allocated_GiB"]=torch.cuda.max_memory_allocated()/2**30
del model;gc.collect();torch.cuda.empty_cache();save_results()
memory("GPU 模型已释放")


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.30it/s]


对照 gpu_bf16 1 {'max_abs': 0.5625, 'mean_abs': 0.048776183277368546, 'KL_a_b': 0.004084252752363682, 'prob_L1': 0.0675874724984169, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 878], 'top5_b': [34, 11439, 53443, 29815, 878]}
对照 gpu_bf16 2 {'max_abs': 0.384765625, 'mean_abs': 0.053641144186258316, 'KL_a_b': 0.004020128399133682, 'prob_L1': 0.046821098774671555, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 66], 'top5_b': [34, 11439, 53443, 29815, 66]}
对照 gpu_bf16 3 {'max_abs': 0.4140625, 'mean_abs': 0.04526097699999809, 'KL_a_b': 0.0035617388784885406, 'prob_L1': 0.06570152938365936, 'argmax_a': 11439, 'argmax_b': 11439, 'top5_a': [11439, 29815, 9173, 984, 2149], 'top5_b': [11439, 29815, 9173, 984, 2149]}
original graph_shared 1 'Copper.' material= True relation= True
original graph_shared 2 'Copper.' material= False relation= False
original graph_shared 3 'According to the provided documents, Project NOVA combines copper supplied by ALPHA and

## 5. 可选 CPU FP32 对照
默认关闭。开启前检查真实可用 RAM（不是机器标称总内存），至少 48 GiB。内存不够则保存跳过原因，不尝试加载。

这里只比较原例三个问题的 K/V、首 token 分布、重复及排列差异，不在 CPU 上执行 72 次长生成。FP32 数值误差大幅下降会支持 BF16 数值解释；若没有下降，则继续审查实现与参考。无论哪种结果，都不能自动证明语义错误由精度造成。


In [11]:
if not RUN_CPU_FP32:
    results["precision_status"]="skipped_by_config"
    print("跳过 CPU FP32；本轮不能声称已完成高精度验证。")
else:
    info=Path("/proc/meminfo")
    available=None
    if info.exists():
        for line in info.read_text().splitlines():
            if line.startswith("MemAvailable:"):available=int(line.split()[1])*1024
    if available is None or available<48*2**30:
        results["precision_status"]="skipped_insufficient_or_unknown_RAM"
        print("可用 RAM 未知或低于 48 GiB，跳过 FP32。")
    else:
        print("CPU FP32 加载，可用 RAM GiB:",available/2**30,flush=True)
        torch.set_num_threads(CPU_THREADS)
        # 纯 CPU 加载；GPU 模型已在上一格释放，不使用自动分配。
        cpu_model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
            torch_dtype=torch.float32,device_map={"":"cpu"},low_cpu_mem_usage=True,
            attn_implementation="eager").eval()
        assert {str(p.device) for p in cpu_model.parameters()}=={"cpu"}
        assert next(cpu_model.parameters()).dtype==torch.float32
        results["precision"].extend(oracle_comparison(cpu_model,"cpu_fp32"))
        results["precision_status"]="completed"
        del cpu_model;gc.collect()
save_results()


CPU FP32 加载，可用 RAM GiB: 237.0340919494629


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.01it/s]


对照 cpu_fp32 1 {'max_abs': 4.863739013671875e-05, 'mean_abs': 3.791133167396765e-06, 'KL_a_b': 1.65890430281479e-08, 'prob_L1': 1.9414382279592246e-07, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 878], 'top5_b': [34, 11439, 53443, 29815, 878]}
对照 cpu_fp32 2 {'max_abs': 3.17692756652832e-05, 'mean_abs': 3.4861247968365205e-06, 'KL_a_b': -5.283474635575658e-08, 'prob_L1': 2.4950079478003317e-06, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 66], 'top5_b': [34, 11439, 53443, 29815, 66]}
对照 cpu_fp32 3 {'max_abs': 3.9577484130859375e-05, 'mean_abs': 3.79516768589383e-06, 'KL_a_b': 6.125684137714416e-08, 'prob_L1': 2.0271038465580205e-06, 'argmax_a': 11439, 'argmax_b': 11439, 'top5_a': [11439, 29815, 9173, 984, 2149], 'top5_b': [11439, 29815, 9173, 984, 2149]}


## 6. 汇总与解释
- 错误跟材料交换而变化：检查模型偏好某种材料，还是偏好某个 source。
- 换项目名/供应商名明显改变结果：名称或分词值得继续研究。
- 反转节点顺序改变答案：研究排列敏感性；这不能排除 BF16 数值因素。
- 图设置的错误稳定出现在 target 2，而顺序输入正确：优先调查第二条对应关系如何被编码和读取。
- FP32 分块/完整图差异明显缩小：更支持 BF16 引起数值差异，仍不能解释全部语义错误。

这些是下一步判断方向，不是预设结论。所有自动检查都需抽查原始回答，尤其含否定或猜测的解释。


In [12]:
import csv
columns=["case","condition","target","project","expected_materials","answer",
         "material_pass","relation_check_pass","wrong_suppliers","eos","hit_limit"]
with (OUTPUT_DIR/"answers.csv").open("w",newline="",encoding="utf-8-sig") as f:
    w=csv.DictWriter(f,fieldnames=columns,extrasaction="ignore");w.writeheader();w.writerows(results["answers"])
for name in CONDITIONS:
    rows=[r for r in results["answers"] if r["condition"]==name]
    print(name,"材料检查",sum(r["material_pass"] for r in rows),"/",len(rows),
          "关系检查",sum(r["relation_check_pass"] for r in rows),"/",len(rows))
print("FP32 状态:",results["precision_status"])
print("结果文件:",OUTPUT_DIR/"graphkv_stage4_results.json")
print("请结合每个变体的原始文本分析，不将这 8 个变体当成独立基准样本。")


graph_shared 材料检查 16 / 24 关系检查 16 / 24
graph_continuous 材料检查 17 / 24 关系检查 17 / 24
ordinary_sequence 材料检查 24 / 24 关系检查 24 / 24
FP32 状态: completed
结果文件: /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_stage4_outputs/graphkv_stage4_results.json
请结合每个变体的原始文本分析，不将这 8 个变体当成独立基准样本。
